# Run everything (one click)

Same as `python run_all.py` in a terminal (this notebook just calls it). Picks the mode by the clock (US Central):

- **Mon / Wed / Fri after 3:15 PM CT** (trading day, first run that day) - **full online update**: company reports (Alpha Vantage rotation, max 24 calls) → news sentiment (NewsAPI 97 calls, never twice within 24 h; Finnhub 97) → earnings dates (Finnhub 97, throttled) → main signal analysis → report check.
- **Any other time** - **quick**: fresh Alpaca daily bars + main signal analysis + report check. No quota APIs.

Ends with a short summary: what ran, API calls used, data date, the holdings **ALERT** and the next check / rebalance.
Put options in `ARGS`, e.g. `["--quick"]`, `["--full"]`, `["--dry-run"]` (plan only), `["--positions", "other.csv"]`.
Add `"--trade"` to auto-trade the Alpaca PAPER account after the pipeline: pulls live positions + equity;
BUY only for buy-signal ('add') symbols, sized Weight × equity (whole shares) net of shares already held —
never above the weight, overweight trimmed with a SELL of the excess; hold-signal symbols are never traded;
sells sell-signal stocks entirely. Submitted as extended-hours DAY limit orders at the closing price
(PAPER only; logs to Reports/paper_orders_log.csv).
If the run reaches the trade step at/after 7:00 PM CT (extended hours over), nothing is submitted —
the planned orders are staged for the next morning's fill check instead. Without `--trade`, no orders are placed.
Safety: `--trade` runs once per scheduled window — a second run is refused until the next Mon/Wed/Fri 3:15 PM CT
trading day, overlapping runs are serialized by Reports/.trade.lock, and SELLs are verified against the live
portfolio (never sold when not held).

In [ ]:
import os
import sys

ARGS = ["--trade"]   # one click: runs all notebooks (full/quick by clock) + auto paper trading

ROOT = os.path.abspath("")          # the notebook's folder = the project folder
sys.path.insert(0, ROOT)
import run_all

exit_code = run_all.main(ARGS)
print("exit code:", exit_code)